# OpenSearch Serverless vs S3 Vectors: Comparison

This notebook visualizes and compares the benchmark results from notebooks 02 and 03.

Comparison areas:
1. **Ingest speed** — Time to store the same vectors
2. **Search latency** — p50/p95/p99 by k value
3. **Feature comparison** — Hybrid search, metadata filtering, cost model, etc.
4. **Workload selection guide**

**Prerequisites:** Complete `02_opensearch_serverless.ipynb` and `03_s3_vectors.ipynb`

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

with open("benchmark_opensearch.json") as f:
    os_data = json.load(f)
with open("benchmark_s3vectors.json") as f:
    s3v_data = json.load(f)

print(f"Benchmark data loaded")
print(f"   OpenSearch: {os_data['vector_count']} vectors, ingest {os_data['ingest_time']:.2f}s")
print(f"   S3 Vectors: {s3v_data['vector_count']} vectors, ingest {s3v_data['ingest_time']:.2f}s")

## 1. Ingest Speed Comparison

Compare the time to store the same vector data in each service.

> **Batch size difference**: OpenSearch bulk API has flexible batch size limits (default 100MB), while S3 Vectors allows up to 500 vectors per request. Optimal batch sizes were used for each service's API characteristics.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
services = ['OpenSearch\nServerless', 'S3 Vectors']
times = [os_data['ingest_time'], s3v_data['ingest_time']]
colors = ['#FF6B35', '#2196F3']

bars = ax.bar(services, times, color=colors, width=0.5, edgecolor='white', linewidth=1.5)
for bar, t, bs in zip(bars, times, [os_data['ingest_batch_size'], s3v_data['ingest_batch_size']]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{t:.2f}s\n(batch={bs})', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Time (seconds)', fontsize=12)
ax.set_title(f'Ingest Speed — {os_data["vector_count"]} vectors', fontsize=14, fontweight='bold')
ax.set_ylim(0, max(times) * 1.4)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 2. Search Latency Comparison

Compare search latency by k value (5, 10, 20). Both p50 (median) and p95 (95th percentile) are shown.

In [ ]:
k_values = ['k=5', 'k=10', 'k=20']
x = np.arange(len(k_values))
width = 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# p50 comparison
os_p50 = [os_data['search'][k]['p50'] for k in k_values]
s3v_p50 = [s3v_data['search'][k]['p50'] for k in k_values]

bars1 = ax1.bar(x - width/2, os_p50, width, label='OpenSearch', color='#FF6B35', edgecolor='white')
bars2 = ax1.bar(x + width/2, s3v_p50, width, label='S3 Vectors', color='#2196F3', edgecolor='white')
for bars in [bars1, bars2]:
    for bar in bars:
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)
ax1.set_xlabel('k value'); ax1.set_ylabel('Latency (ms)')
ax1.set_title('p50 (Median) Latency', fontsize=13, fontweight='bold')
ax1.set_xticks(x); ax1.set_xticklabels(k_values)
ax1.legend(); ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

# p95 comparison
os_p95 = [os_data['search'][k]['p95'] for k in k_values]
s3v_p95 = [s3v_data['search'][k]['p95'] for k in k_values]

bars1 = ax2.bar(x - width/2, os_p95, width, label='OpenSearch', color='#FF6B35', edgecolor='white')
bars2 = ax2.bar(x + width/2, s3v_p95, width, label='S3 Vectors', color='#2196F3', edgecolor='white')
for bars in [bars1, bars2]:
    for bar in bars:
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)
ax2.set_xlabel('k value'); ax2.set_ylabel('Latency (ms)')
ax2.set_title('p95 (Tail) Latency', fontsize=13, fontweight='bold')
ax2.set_xticks(x); ax2.set_xticklabels(k_values)
ax2.legend(); ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

plt.suptitle(f'Search Latency — {os_data["vector_count"]} vectors', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Latency summary table
print(f"{'Service':<25} {'k=5 p50':>10} {'k=5 p95':>10} {'k=10 p50':>10} {'k=20 p50':>10}")
print('-' * 70)
for data, name in [(os_data, 'OpenSearch Serverless'), (s3v_data, 'S3 Vectors')]:
    print(f"{name:<25} {data['search']['k=5']['p50']:>9.1f}ms {data['search']['k=5']['p95']:>9.1f}ms "
          f"{data['search']['k=10']['p50']:>9.1f}ms {data['search']['k=20']['p50']:>9.1f}ms")

## 3. Workload Selection Guide

The two services have different design philosophies, so the best choice depends on workload characteristics.

### Choose OpenSearch Serverless when:
- You need **keyword search** on video metadata (title, description, tags) alongside embedding-based **semantic search**
- **Hybrid search** is needed for more accurate ranking
- Search latency under **25ms** is critical for real-time services
- You want to directly tune **index algorithms** like HNSW, IVF

### Choose S3 Vectors when:
- You want to **get started quickly** (just create Vector Bucket + Index)
- **Cost efficiency** matters for small-scale workloads (no minimum OCU cost)
- You need vector **upsert** (automatic update on re-generation)
- You want natural integration with existing **S3 infrastructure**
- Only **pure vector similarity search** is needed without full-text search

## Summary

In this workshop, we stored and searched video embeddings generated by TwelveLabs Marengo 3.0 using two AWS vector stores.

**Key takeaways:**
- **OpenSearch Serverless** — Best when hybrid search (vector + keyword) is needed and low search latency is critical
- **S3 Vectors** — Best when simple setup and cost efficiency matter, and pure vector search is sufficient

For production adoption, we recommend running a PoC with your own workload data and query patterns. The vector count in this workshop (hundreds to thousands) is a small-scale test — results may differ at scale (hundreds of thousands to millions of vectors) due to each service's scaling characteristics.